<a href="https://colab.research.google.com/github/Parth-1104/FastAPI-GEN-AI/blob/main/projectTSA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
df=pd.read_csv('/content/Battery_dataset 2.csv')
df.head()

,battery_id,cycle,chI,chV,chT,disI,disV,disT,BCt,SOH,RUL
0,B5,1,1.440147,4.254682,23.988733,1.894407,3.273523,32.980834,1.986196,99.309790,219
1,B5,2,1.416595,4.159825,25.665347,1.829949,4.038741,32.257920,1.986240,99.311985,218
2,B5,3,1.420272,4.276323,25.407910,1.942105,3.214433,35.134801,1.984252,99.212608,217
3,B5,4,1.337680,4.236697,27.069757,2.073577,3.134529,32.082988,1.969236,98.461812,216
4,B5,5,1.263946,4.142791,26.478353,2.049885,3.729341,32.483154,1.974862,98.743106,215


In [8]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Conv1D, BatchNormalization, Activation, GRU, Dropout, Dense, Input, Flatten
from tensorflow.keras.models import Model
from scipy.optimize import curve_fit

def create_cnn_gru_model(input_shape):
    from tensorflow.keras.layers import Lambda
    inputs = Input(shape=input_shape)
    x = Conv1D(10, 3, padding='same')(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Flatten()(x)
    x = Lambda(lambda t: tf.expand_dims(t, axis=1))(x)  # Fix here
    x = GRU(60, activation='tanh')(x)
    x = Dropout(0.2)(x)
    outputs = Dense(1)(x)
    model = Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(0.005), loss='mse')
    return model


def double_exp(k, a, b, c, d):
    return a * np.exp(b * k) + c * np.exp(d * k)

class ParticleFilter:
    def __init__(self, num_particles, param_init, noise_std=1e-4):
        self.N = num_particles
        self.params = np.array([param_init] * self.N, dtype=float)
        self.weights = np.ones(self.N) / self.N
        self.noise_std = noise_std

    def predict(self):
        self.params += np.random.normal(0, self.noise_std, self.params.shape)

    def update(self, k, observation):
        preds = np.array([double_exp(k, *p) for p in self.params])
        likelihoods = np.exp(-0.5 * ((observation - preds) ** 2) / (self.noise_std ** 2))
        self.weights *= likelihoods
        self.weights += 1e-300
        self.weights /= np.sum(self.weights)
        cumulative_sum = np.cumsum(self.weights)
        cumulative_sum[-1] = 1.
        indexes = np.searchsorted(cumulative_sum, np.random.rand(self.N))
        self.params = self.params[indexes]
        self.weights.fill(1.0 / self.N)

    def estimate(self, k):
        est_param = np.average(self.params, axis=0, weights=self.weights)
        estimate = double_exp(k, *est_param)
        return estimate, est_param

def battery_rul_prediction_from_user_input(user_inputs, window_size=20, pretrained_model=None):
    # user_inputs is a dictionary containing keys: cycle, voltage, current, temperature, capacity, window_data (array)
    window_data = np.array(user_inputs['window_data'])

    # Prepare rolling window training data from window_data (small sliding windows)
    X = np.array([window_data[i:i+window_size-1] for i in range(len(window_data)-window_size+1)])
    y = np.array([window_data[i+window_size-1] for i in range(len(window_data)-window_size+1)])
    X = X[..., np.newaxis]

    if pretrained_model is None:
        # Build model and train on this small window dataset if pretrained model not provided
        model = create_cnn_gru_model((window_size-1, 1))
        model.fit(X, y, epochs=80, batch_size=8, verbose=0)
    else:
        model = pretrained_model

    try:
        popt, _ = curve_fit(double_exp, np.arange(window_size-1), window_data[:-1], maxfev=4000)
        # Bounds can be added here to avoid overflow
    except:
        popt = [window_data[-1], -0.01, window_data[-1]/2, -0.01]

    pf = ParticleFilter(300, popt, 0.0005)
    current_series = window_data.copy()
    predictions = []
    failure_threshold = 0.7 * user_inputs['capacity']

    for k in range(window_size-1, window_size+200):
        xin = current_series[-(window_size-1):].reshape(1, window_size-1, 1)
        pred_cap = float(model.predict(xin, verbose=0)[0,0])
        pf.predict()
        pf.update(k, pred_cap)
        pf_cap, _ = pf.estimate(k)
        current_series = np.append(current_series[1:], pf_cap)
        predictions.append(pf_cap)

        X_new = np.array([current_series[i:i+window_size-1] for i in range(len(current_series)-(window_size-1))])
        y_new = np.array([current_series[i+window_size-1] for i in range(len(current_series)-(window_size-1))])
        X_new = X_new[..., np.newaxis]
        model.fit(X_new, y_new, epochs=5, batch_size=8, verbose=0)

        if pf_cap <= failure_threshold:
            break

    rul_cycles = len(predictions)
    soh_estimated = pf_cap / user_inputs['capacity'] * 100

    return round(soh_estimated, 2), rul_cycles



    # Train with ALL valid rolling windows from the historical series
    full_window = battery_df['BCt'].values
    Xfull = np.array([full_window[i:i+window_size-1] for i in range(len(full_window)-window_size)])
    yfull = np.array([full_window[i+window_size-1] for i in range(len(full_window)-window_size)])
    Xfull = Xfull[..., np.newaxis]

    model = create_cnn_gru_model((window_size-1, 1))
    model.fit(Xfull, yfull, epochs=80, batch_size=8, verbose=0)

    try:
        popt, _ = curve_fit(double_exp, np.arange(window_size-1), window_data[:-1], maxfev=4000)
    except:
        popt = [window_data[-1], -0.01, window_data[-1]/2, -0.01]

    pf = ParticleFilter(300, popt, 0.0005)
    current_series = window_data.copy()
    predictions = []
    failure_threshold = 0.7 * user_inputs['capacity']

    for k in range(window_size-1, window_size+200):
        xin = current_series[-(window_size-1):].reshape(1, window_size-1, 1)
        pred_cap = float(model.predict(xin, verbose=0)[0,0])
        pf.predict()
        pf.update(k, pred_cap)
        pf_cap, _ = pf.estimate(k)
        current_series = np.append(current_series[1:], pf_cap)
        predictions.append(pf_cap)

        # Iterative window retraining (optional)
        X_new = np.array([current_series[i:i+window_size-1] for i in range(len(current_series)-(window_size-1))])
        y_new = np.array([current_series[i+window_size-1] for i in range(len(current_series)-(window_size-1))])
        X_new = X_new[..., np.newaxis]
        model.fit(X_new, y_new, epochs=5, batch_size=8, verbose=0)

        if pf_cap <= failure_threshold:
            break

    soh_estimated = pf_cap / user_inputs['capacity'] * 100
    rul_cycles = len(predictions)
    return round(soh_estimated, 2), rul_cycles


user_inputs = {
    'cycle': 150,
    'voltage': 3.7,
    'current': 1.0,
    'temperature': 30.0,
    'capacity': 2800,
    'window_data': [
        2950, 2920, 2890, 2860, 2830, 2800, 2770, 2740, 2710, 2680,
        2650, 2620, 2590, 2560, 2530, 2500, 2470, 2440, 2410, 2380
    ]
}

soh, rul = battery_rul_prediction_from_user_input(user_inputs)
print(f"Estimated SOH: {soh}%")
print(f"Estimated RUL: {rul} cycles")



Estimated SOH: 68.38%
Estimated RUL: 41 cycles


In [13]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Conv1D, BatchNormalization, Activation, GRU, Dropout, Dense, Input, Flatten, Lambda
from tensorflow.keras.models import Model
from scipy.optimize import curve_fit

# Define CNN-GRU model architecture
def create_cnn_gru_model(input_shape):
    inputs = Input(shape=input_shape)
    x = Conv1D(10, 3, padding='same')(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Flatten()(x)
    x = Lambda(lambda t: tf.expand_dims(t, axis=1))(x)  # Add temporal dimension for GRU
    x = GRU(60, activation='tanh')(x)
    x = Dropout(0.2)(x)
    outputs = Dense(1)(x)
    model = Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(0.005), loss='mse')
    return model

# Double exponential function used in particle filter
def double_exp(k, a, b, c, d):
    return a * np.exp(b * k) + c * np.exp(d * k)

# Particle Filter class for parameter fusion
class ParticleFilter:
    def __init__(self, num_particles, param_init, noise_std=1e-4):
        self.N = num_particles
        self.params = np.array([param_init] * self.N, dtype=float)
        self.weights = np.ones(self.N) / self.N
        self.noise_std = noise_std

    def predict(self):
        self.params += np.random.normal(0, self.noise_std, self.params.shape)

    def update(self, k, observation):
        preds = np.array([double_exp(k, *p) for p in self.params])
        likelihoods = np.exp(-0.5 * ((observation - preds) ** 2) / (self.noise_std ** 2))
        self.weights *= likelihoods
        self.weights += 1e-300
        self.weights /= np.sum(self.weights)
        cumulative_sum = np.cumsum(self.weights)
        cumulative_sum[-1] = 1.
        indexes = np.searchsorted(cumulative_sum, np.random.rand(self.N))
        self.params = self.params[indexes]
        self.weights.fill(1.0 / self.N)

    def estimate(self, k):
        est_param = np.average(self.params, axis=0, weights=self.weights)
        estimate = double_exp(k, *est_param)
        return estimate, est_param

# Training function on large dataset
def train_model_on_dataset(df, window_size=20, epochs=80, batch_size=8, model_weights_path='cnn_gru_battery.weights.h5'):
    # Use single battery example or loop for all batteries to get bigger data
    battery_id = df['battery_id'].iloc[0]
    battery_df = df[df['battery_id'] == battery_id].reset_index(drop=True)
    full_window = battery_df['BCt'].values

    # Prepare windowed training sequences and labels
    X = np.array([full_window[i:i+window_size-1] for i in range(len(full_window)-window_size)])
    y = np.array([full_window[i+window_size-1] for i in range(len(full_window)-window_size)])
    X = X[..., np.newaxis]

    model = create_cnn_gru_model((window_size-1, 1))
    model.fit(X, y, epochs=epochs, batch_size=batch_size, verbose=1)
    model.save_weights(model_weights_path)
    print(f"Model trained and saved at {model_weights_path}")
    return model


# Prediction function using pretrained model and user input
def predict_soh_rul(user_inputs, window_size=20, model_weights_path='/content/cnn_gru_battery.weights.h5'):
    window_data = np.array(user_inputs['window_data'])

    # Prepare input windows
    X = np.array([window_data[i:i+window_size-1] for i in range(len(window_data)-window_size+1)])
    y = np.array([window_data[i+window_size-1] for i in range(len(window_data)-window_size+1)])
    X = X[..., np.newaxis]

    # Load pretrained model weights
    model = create_cnn_gru_model((window_size-1,1))
    model.load_weights(model_weights_path)

    try:
        popt, _ = curve_fit(double_exp, np.arange(window_size-1), window_data[:-1], maxfev=4000,
                            bounds=([0, -np.inf, 0, -np.inf], [np.inf, 0, np.inf, 0]))
    except:
        popt = [window_data[-1], -0.01, window_data[-1]/2, -0.01]

    pf = ParticleFilter(300, popt, 0.0005)
    current_series = window_data.copy()
    predictions = []

    failure_threshold = 0.7 * user_inputs['capacity']

    for k in range(window_size-1, window_size+200):
        xin = current_series[-(window_size-1):].reshape(1, window_size-1, 1)
        pred_cap = float(model.predict(xin, verbose=0)[0,0])
        pf.predict()
        pf.update(k, pred_cap)
        pf_cap, _ = pf.estimate(k)
        current_series = np.append(current_series[1:], pf_cap)
        predictions.append(pf_cap)

        # Optional iterative retraining can be disabled for speed
        # X_new = np.array([current_series[i:i+window_size-1] for i in range(len(current_series)-(window_size-1))])
        # y_new = np.array([current_series[i+window_size-1] for i in range(len(current_series)-(window_size-1))])
        # X_new = X_new[..., np.newaxis]
        # model.fit(X_new, y_new, epochs=5, batch_size=8, verbose=0)

        if pf_cap <= failure_threshold:
            break

    rul_cycles = len(predictions)
    soh_estimated = pf_cap / user_inputs['capacity'] * 100

    return round(soh_estimated, 2), rul_cycles



In [14]:
# --- Usage Example ---

# Load your dataset CSV file here
df = pd.read_csv('/content/Battery_dataset 2.csv')

# Step 1: Train model on dataset (run once)
model = train_model_on_dataset(df, window_size=20, epochs=80, batch_size=8)

# Step 2: Prepare user input for prediction (could come from battery specs or user measurement)


Epoch 1/80
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 0.7256
Epoch 2/80
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0624
Epoch 3/80
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0542
Epoch 4/80
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0401
Epoch 5/80
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0269
Epoch 6/80
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0570
Epoch 7/80
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0362
Epoch 8/80
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0443
Epoch 9/80
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0324
Epoch 10/80
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0234
Epoch 11/80
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0287
Epoch 12/80
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0384
Epoch 13/80
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0353
Epoch 14/80
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0320
Epoch 15/80
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0294
Epoch 16/80
25/25 ━

In [16]:
user_inputs = {
    'cycle': 150,
    'voltage': 3.7,
    'current': 1.0,
    'temperature': 30.0,
    'capacity': 2800,
    'window_data': [
        2950, 2920, 2890, 2860, 2830, 2800, 2770, 2740, 2710, 2680,
        2650, 2620, 2590, 2560, 2530, 2500, 2470, 2440, 2410, 2380
    ]
}

# Step 3: Predict using pretrained model
soh, rul = predict_soh_rul(user_inputs, window_size=20, model_weights_path='/content/cnn_gru_battery.weights.h5')
print(f"Estimated State of Health (SOH): {soh}%")
print(f"Estimated Remaining Useful Life (RUL): {rul} cycles")


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 20 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Estimated State of Health (SOH): 68.75%
Estimated Remaining Useful Life (RUL): 20 cycles


In [ ]:
from google.colab import drive
drive.mount('/content/drive')